In [2]:
import sqlite3, pandas as pd
df = pd.read_csv("data/clean.csv")
con = sqlite3.connect(":memory:")
df.to_sql("t", con, index=False, if_exists="replace")
def q(sql):
    return pd.read_sql_query(sql, con)
q("SELECT COUNT(*) AS rows FROM t")

,rows
0,30000


In [5]:
q("SELECT AVG(CASE WHEN delivery_status <> 'delivered' THEN 1.0 ELSE 0 END) * 100 AS fail_pct FROM t")

,fail_pct
0,8.553333


In [8]:
q("SELECT network_provider, AVG(CASE WHEN delivery_status <> 'delivered' THEN 1.0 ELSE 0 END) AS fail_rate, COUNT(*) AS n FROM t GROUP BY network_provider ORDER BY fail_rate DESC")

,network_provider,fail_rate,n
0,Provider_D,0.147191,2901
1,Provider_C,0.099931,5794
2,Unknown Provider,0.098616,578
3,Provider_B,0.079678,8823
4,Provider_A,0.067204,11904


In [17]:
q("SELECT strftime('%H', sent_at) AS HOUR, AVG(CASE WHEN delivery_status <> 'delivered' THEN 1.0 ELSE 0 END) AS fail_rate, COUNT(*) AS n FROM t GROUP BY hour ORDER BY fail_rate DESC")

,HOUR,fail_rate,n
0,21,0.176471,221
1,22,0.174468,235
2,19,0.165803,193
3,18,0.149864,367
4,01,0.146341,615
5,23,0.134454,238
6,20,0.120773,207
7,00,0.102908,447
8,04,0.089194,1166
9,13,0.087459,2424


In [19]:
q("WITH prov AS(SELECT network_provider, AVG(CASE WHEN delivery_status <> 'delivered' THEN 1.0 ELSE 0 END) AS fail_rate, COUNT(*) AS n FROM t GROUP BY network_provider) SELECT network_provider, fail_rate, n, RANK() OVER(ORDER BY fail_rate DESC) AS worst_rank FROM prov ORDER BY worst_rank")

,network_provider,fail_rate,n,worst_rank
0,Provider_D,0.147191,2901,1
1,Provider_C,0.099931,5794,2
2,Unknown Provider,0.098616,578,3
3,Provider_B,0.079678,8823,4
4,Provider_A,0.067204,11904,5
